# Progetto Machine Learning: Riconoscimento di Specie di Uccelli con CNN

Questo notebook implementa un sistema di riconoscimento di specie di uccelli attraverso l'analisi di registrazioni audio della competizione BirdClef 2025. Il progetto utilizza un'architettura CNN per classificare gli audio convertiti in spettrogrammi Mel e include anche un sistema di configurazione automatica dell'ambiente per eseguire il codice su Kaggle, Google Colab o in locale.

## 1. Importazione delle Librerie Necessarie

Importiamo tutte le librerie necessarie per l'elaborazione audio, deep learning e visualizzazione.

In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install --no-cache-dir audiomentations --use-deprecated=legacy-resolver

In [ ]:
# Librerie di sistema e utilità
import os
import sys
import platform
import time
import warnings
import logging
import datetime
from pathlib import Path
import pprint as pp
import seaborn as sns
from collections import Counter
import IPython.display as ipd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from audiomentations import BandStopFilter, Shift, TimeMask, AddGaussianNoise
# Sostituisci le importazioni di Transformers con timm
import timm
import math
import random
import soundfile as sf

# Librerie per data science e manipolazione dati
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split

# Librerie per elaborazione audio
import librosa
import librosa.display

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
from torchvision import transforms

# Visualizzazione
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Ignoriamo i warning
warnings.filterwarnings("ignore")

# Configurazione del logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('BirdClef')


# Ottimizzazione dell'utilizzo della CPU per PyTorch
if hasattr(torch, 'set_num_threads') and os.cpu_count() > 4:
    torch.set_num_threads(os.cpu_count())
    print(f"PyTorch configurato per utilizzare {os.cpu_count()} CPU threads")

# Se hai accesso a MKL (per operazioni su CPU)
import os
os.environ['MKL_NUM_THREADS'] = str(os.cpu_count())
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count())

print("Librerie importate con successo!")
print(f"PyTorch versione: {torch.__version__}")
print(f"timm versione: {timm.__version__}")
print(f"Python versione: {platform.python_version()}")
print(f"Sistema operativo: {platform.system()} {platform.release()}")

Librerie importate con successo!
PyTorch versione: 2.5.1+cu124
timm versione: 1.0.14
Python versione: 3.11.11
Sistema operativo: Linux 6.6.56+


In [2]:
import shutil
import os

# Imposta questo a True per abilitare la cancellazione
clear_working_dir = True

working_dir = '/kaggle/working/'

if clear_working_dir:
    for filename in os.listdir(working_dir):
        file_path = os.path.join(working_dir, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)  # elimina file o link
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)  # elimina directory
        except Exception as e:
            print(f'Errore durante la rimozione di {file_path}: {e}')
    print(f"Tutti i file in {working_dir} sono stati rimossi.")
else:
    print("Pulizia disabilitata (clear_working_dir = False)")


Tutti i file in /kaggle/working/ sono stati rimossi.


In [ ]:
class Config:
    def __init__(self):
        
        # Imposta i percorsi di base in base all'ambiente
        self.COMPETITION_NAME = "birdclef-2025"
        self.BASE_DIR = f"/kaggle/input/{self.COMPETITION_NAME}"
        self.OUTPUT_DIR = "/kaggle/working"
        self.MODELS_DIR = "/kaggle/input"  # Per i modelli pre-addestrati
            
        # Imposta subito i percorsi derivati per l'ambiente Kaggle
        self._setup_derived_paths()
            
        
        # Parametri per il preprocessing audio - già allineati con vincitori
        self.SR = 32000      # Sample rate
        self.DURATION = 5    # Durata dei clip in secondi
        self.N_MELS = 224    # Numero di bande Mel
        self.N_FFT = 2048    # Dimensione finestra FFT
        self.HOP_LENGTH = 512  # Hop length per STFT
        self.FMIN = 48       # Frequenza minima per lo spettrogramma Mel
        self.FMAX = 16000    # Frequenza massima
        self.POWER = 2       # Esponente per calcolo spettrogramma
            
        # Parametri per il training - aggiornati secondo i vincitori
        self.BATCH_SIZE = 96  # Aumentato da 32 a 96 come dai vincitori
        self.EPOCHS = 15     # Numero di epoche per il training
        self.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        self.NUM_WORKERS = 4  # Aumentato per migliorare il data loading

        # Parametri per inference/submission
        self.TEST_CLIP_DURATION = 5  # Durata dei segmenti per la predizione (secondi)
        self.N_CLASSES = 0  # Sarà impostato dopo aver caricato i dati

    def _setup_derived_paths(self):
        """Imposta i percorsi derivati basati su BASE_DIR"""
        # Utilizza la normale divisione di percorso di OS (non il backslash hardcoded)
        self.TRAIN_AUDIO_DIR = os.path.join(self.BASE_DIR, "train_audio")
        self.TEST_SOUNDSCAPES_DIR = os.path.join(self.BASE_DIR, "test_soundscapes")
        self.TRAIN_CSV_PATH = os.path.join(self.BASE_DIR, "train.csv")
        self.TAXONOMY_CSV_PATH = os.path.join(self.BASE_DIR, "taxonomy.csv") 
        self.SAMPLE_SUB_PATH = os.path.join(self.BASE_DIR, "sample_submission.csv")

In [ ]:
config = Config()

# Stampa percorsi aggiornati
print(f"\nPercorso file CSV di training: {config.TRAIN_CSV_PATH}")
print(f"Percorso directory audio di training: {config.TRAIN_AUDIO_DIR}")


Percorso file CSV di training: /kaggle/input/birdclef-2025/train.csv
Percorso directory audio di training: /kaggle/input/birdclef-2025/train_audio


In [ ]:
# Caricamento dei metadati
def load_metadata():
    """
    Carica e prepara i metadati dal file CSV di training.
    
    Returns:
        tuple: training_df, all_species, labels_one_hot
    """
    print(f"Caricamento metadati da: {config.TRAIN_CSV_PATH}")
    train_df = pd.read_csv(config.TRAIN_CSV_PATH)
    sample_sub_df = pd.read_csv(config.SAMPLE_SUB_PATH)
    
    # Estrai tutte le etichette uniche
    train_primary_labels = train_df['primary_label'].unique()
    train_secondary_labels = set([lbl for sublist in train_df['secondary_labels'].apply(eval) 
                                 for lbl in sublist if lbl])
    submission_species = sample_sub_df.columns[1:].tolist()  # Escludi row_id
    
    # Combina tutte le possibili etichette
    all_species = sorted(list(set(train_primary_labels) | train_secondary_labels | set(submission_species)))
    N_CLASSES = len(all_species)
    config.N_CLASSES = N_CLASSES  # Aggiorna il numero di classi nella configurazione
    
    print(f"Numero totale di specie trovate: {N_CLASSES}")
    print(f"Prime 10 specie: {all_species[:10]}")
    
    # Crea mappatura etichette-indici
    species_to_int = {species: i for i, species in enumerate(all_species)}
    int_to_species = {i: species for species, i in species_to_int.items()}
    
    # Aggiungi indici numerici al dataframe
    train_df['primary_label_int'] = train_df['primary_label'].map(species_to_int)
    
    # Prepara target multi-etichetta
    mlb = MultiLabelBinarizer(classes=all_species)
    mlb.fit(None)  # Fit con tutte le classi
    
    def get_multilabel(row):
        labels = eval(row['secondary_labels'])  # Valuta la lista di stringhe in modo sicuro
        labels.append(row['primary_label'])
        return list(set(labels))  # Assicura etichette uniche
    
    train_df['all_labels'] = train_df.apply(get_multilabel, axis=1)
    train_labels_one_hot = mlb.transform(train_df['all_labels'])
    
    print(f"Forma delle etichette one-hot: {train_labels_one_hot.shape}")
    
    return train_df, all_species, train_labels_one_hot, species_to_int, int_to_species

In [ ]:
# Carica i metadati
train_df, all_species, train_labels_one_hot, species_to_int, int_to_species = load_metadata()

## Analisi Esplorativa dei Dati (EDA)

In questa sezione esploreremo le caratteristiche del dataset per comprendere meglio la distribuzione delle specie, le proprietà audio e identificare eventuali pattern nei dati.

In [ ]:
# Configurazione stile visualizzazioni
plt.style.use('seaborn-v0_8-whitegrid')  # Aggiornamento per compatibilità con nuove versioni
sns.set(style="whitegrid", font_scale=1.2)
plt.rcParams['figure.figsize'] = [14, 7]
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

print("=== 📊 Statistiche di base del dataset BirdClef ===")
print(f"📈 Numero totale di registrazioni: {len(train_df):,}")
print(f"🦜 Numero di specie uniche nel dataset: {len(all_species):,}")

# 1. PANORAMICA DELLA DISTRIBUZIONE DEGLI ESEMPI PER CLASSE
primary_species_count = train_df['primary_label'].value_counts()
mean_examples = primary_species_count.mean()
median_examples = primary_species_count.median()

print("\n=== 📊 Distribuzione degli esempi per classe ===")
print(f"🔹 Media esempi per specie: {mean_examples:.1f}")
print(f"🔹 Mediana esempi per specie: {median_examples:.0f}")
print(f"🔹 Massimo numero di esempi: {primary_species_count.max():,} ({primary_species_count.idxmax()})")
print(f"🔹 Minimo numero di esempi: {primary_species_count.min():,} ({primary_species_count.idxmin()})")

# Calcola quantili importanti
q1 = primary_species_count.quantile(0.25)
q3 = primary_species_count.quantile(0.75)
print(f"🔹 1° quartile (25%): {q1:.0f} esempi")
print(f"🔹 3° quartile (75%): {q3:.0f} esempi")

# 2. VISUALIZZAZIONE AVANZATA DELLA DISTRIBUZIONE
fig, axes = plt.subplots(2, 2, figsize=(16, 12), gridspec_kw={'height_ratios': [1, 1.2]})

# 2.1 Top 15 specie più rappresentate
top_species = primary_species_count.head(15)
bars = sns.barplot(x=top_species.index, y=top_species.values, ax=axes[0, 0], palette="viridis")
axes[0, 0].set_title('Top 15 Specie più Rappresentate', fontweight='bold')
axes[0, 0].set_xlabel('Specie')
axes[0, 0].set_ylabel('Numero di Registrazioni')
axes[0, 0].tick_params(axis='x', rotation=90)

# Annota il numero di esempi sopra ogni barra
for i, bar in enumerate(bars.patches):
    axes[0, 0].text(
        bar.get_x() + bar.get_width()/2, 
        bar.get_height() + 5, 
        f'{int(bar.get_height())}', 
        ha='center', va='bottom',
        fontsize=10, color='black'
    )

# 2.2 15 specie meno rappresentate
bottom_species = primary_species_count.tail(15)
bars = sns.barplot(x=bottom_species.index, y=bottom_species.values, ax=axes[0, 1], palette="rocket")
axes[0, 1].set_title('15 Specie meno Rappresentate', fontweight='bold')
axes[0, 1].set_xlabel('Specie')
axes[0, 1].set_ylabel('Numero di Registrazioni')
axes[0, 1].tick_params(axis='x', rotation=90)

# Annota il numero di esempi sopra ogni barra
for i, bar in enumerate(bars.patches):
    axes[0, 1].text(
        bar.get_x() + bar.get_width()/2, 
        bar.get_height() + 0.1, 
        f'{int(bar.get_height())}', 
        ha='center', va='bottom',
        fontsize=10, color='black'
    )

# 2.3 Distribuzione complessiva (boxplot + swarm)
sns.boxplot(x=primary_species_count.values, ax=axes[1, 0], color='lightblue', width=0.3)
axes[1, 0].set_title('Distribuzione degli Esempi per Specie (Boxplot)', fontweight='bold')
axes[1, 0].set_xlabel('Numero di Esempi')
axes[1, 0].set_ylabel('Densità')

# Aggiungi linee verticali per media e mediana
axes[1, 0].axvline(mean_examples, color='red', linestyle='--', label=f'Media: {mean_examples:.1f}')
axes[1, 0].axvline(median_examples, color='green', linestyle='--', label=f'Mediana: {median_examples:.0f}')
axes[1, 0].legend()

# 2.4 Istogramma con curve di densità
sns.histplot(primary_species_count.values, bins=30, kde=True, ax=axes[1, 1], color='skyblue')
axes[1, 1].set_title('Distribuzione degli Esempi per Specie (Istogramma)', fontweight='bold')
axes[1, 1].set_xlabel('Numero di Esempi')
axes[1, 1].set_ylabel('Numero di Specie')

# Aggiungi linee verticali per media, mediana e varie soglie di interesse
axes[1, 1].axvline(mean_examples, color='red', linestyle='--', label=f'Media: {mean_examples:.1f}')
axes[1, 1].axvline(median_examples, color='green', linestyle='--', label=f'Mediana: {median_examples:.0f}')
axes[1, 1].axvline(40, color='orange', linestyle=':', label='Soglia 40 esempi')
axes[1, 1].axvline(100, color='purple', linestyle=':', label='Soglia 100 esempi')
axes[1, 1].axvline(5, color='brown', linestyle=':', label='Classi estremamente rare (≤5)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# 3. ANALISI DELLO SBILANCIAMENTO DEL DATASET
# Calcolo dell'indice di Gini e visualizzazione della Curva di Lorenz
def gini_coefficient(x):
    x = np.sort(x)
    n = len(x)
    index = np.arange(1, n+1)
    return (np.sum((2*index - n - 1) * x)) / (n * np.sum(x))

def plot_lorenz_curve(x):
    x_lorenz = np.sort(x)
    x_lorenz = np.cumsum(x_lorenz)
    x_lorenz = x_lorenz / x_lorenz[-1]
    return x_lorenz

gini = gini_coefficient(primary_species_count.values)

plt.figure(figsize=(10, 6))
# Curva di Lorenz
lorenz_curve = plot_lorenz_curve(primary_species_count.values)
plt.plot(np.linspace(0, 1, len(lorenz_curve)), lorenz_curve, label='Distribuzione degli esempi', color='blue')
# Linea di perfetta uguaglianza
plt.plot([0, 1], [0, 1], 'k--', label='Distribuzione ideale (bilanciata)')
# Area di Gini colorata
plt.fill_between(np.linspace(0, 1, len(lorenz_curve)), np.linspace(0, 1, len(lorenz_curve)), lorenz_curve, alpha=0.2, color='red')

plt.text(0.6, 0.3, f'Indice di Gini: {gini:.4f}', fontsize=14, bbox=dict(facecolor='white', alpha=0.8))
plt.text(0.6, 0.2, f'Sbilanciamento: {"Alto" if gini > 0.6 else "Moderato" if gini > 0.3 else "Basso"}', 
         fontsize=14, bbox=dict(facecolor='white', alpha=0.8))

plt.title('Curva di Lorenz: Visualizzazione dello Sbilanciamento del Dataset', fontweight='bold')
plt.xlabel('Percentile cumulativo delle specie')
plt.ylabel('Percentile cumulativo degli esempi')
plt.grid(True)
plt.legend()
plt.show()

# 4. ANALISI DELLE CLASSI RARE E PROBLEMATICHE PER IL MODELLO
rare_thresholds = {
    "estremamente_rare": 5,    # <= 5 esempi
    "molto_rare": 20,          # <= 20 esempi
    "rare": 40,                # <= 40 esempi
    "mediamente_rare": 100     # <= 100 esempi
}

# Crea un dataframe che contiene le statistiche sulle classi
rare_stats = pd.DataFrame({
    'threshold': rare_thresholds.values(),
    'threshold_name': rare_thresholds.keys(),
    'num_classes': [sum(primary_species_count <= th) for th in rare_thresholds.values()],
    'perc_classes': [sum(primary_species_count <= th)/len(primary_species_count)*100 for th in rare_thresholds.values()]
})

# Crea il DataFrame per la visualizzazione
rare_stats['perc_formatted'] = rare_stats['perc_classes'].apply(lambda x: f'{x:.1f}%')
rare_stats['label'] = rare_stats.apply(lambda x: f"{x['num_classes']} classi\n({x['perc_formatted']})", axis=1)

print("\n=== 🚨 Analisi delle Classi Rare ===")
print(f"Numero totale di classi: {len(primary_species_count):,}")
for idx, row in rare_stats.iterrows():
    print(f"🔸 Classi con ≤{row['threshold']} esempi ({row['threshold_name']}): {row['num_classes']} ({row['perc_formatted']})")

# Visualizza le classi rare come grafico ad anelli
plt.figure(figsize=(10, 8))
colors = ['#fc8d59', '#fee090', '#91bfdb', '#4575b4', '#ffffbf']

sizes = [
    rare_stats.iloc[0]['num_classes'],
    rare_stats.iloc[1]['num_classes'] - rare_stats.iloc[0]['num_classes'],
    rare_stats.iloc[2]['num_classes'] - rare_stats.iloc[1]['num_classes'],
    rare_stats.iloc[3]['num_classes'] - rare_stats.iloc[2]['num_classes'],
    len(primary_species_count) - rare_stats.iloc[3]['num_classes']
]

labels = [
    f"≤{rare_thresholds['estremamente_rare']} esempi: {rare_stats.iloc[0]['num_classes']} classi",
    f"{rare_thresholds['estremamente_rare']+1}-{rare_thresholds['molto_rare']} esempi: {sizes[1]} classi",
    f"{rare_thresholds['molto_rare']+1}-{rare_thresholds['rare']} esempi: {sizes[2]} classi",
    f"{rare_thresholds['rare']+1}-{rare_thresholds['mediamente_rare']} esempi: {sizes[3]} classi",
    f">100 esempi: {sizes[4]} classi"
]

# Plot con wedge con separazione per evidenziare i segmenti
wedges, texts, autotexts = plt.pie(
    sizes, 
    labels=labels,
    colors=colors, 
    autopct='%1.1f%%',
    startangle=90,
    explode=(0.1, 0.05, 0, 0, 0),  # Esplodi il primo elemento (più critico)
    shadow=True,
    textprops={'fontsize': 12},
    wedgeprops={'linewidth': 0.7, 'edgecolor': 'white'}
)

# Personalizza il testo
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_color('black')

plt.axis('equal')
plt.title('Distribuzione delle Classi per Numero di Esempi', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

# 5. CONCLUSIONI E IMPLICAZIONI PER IL MACHINE LEARNING

print("\n=== 🎯 Conclusioni per il Modello di Machine Learning ===")
print("1️⃣ Il dataset presenta un forte sbilanciamento (indice di Gini elevato)")
print(f"2️⃣ Le classi con ≤{rare_thresholds['rare']} esempi ({rare_stats.iloc[2]['num_classes']} classi, {rare_stats.iloc[2]['perc_formatted']}) richiedono tecniche di oversampling o data augmentation")
print(f"3️⃣ Le classi estremamente rare (≤{rare_thresholds['estremamente_rare']} esempi) potrebbero beneficiare di tecniche avanzate come transfer learning o few-shot learning")
print("4️⃣ È consigliabile monitorare separatamente le metriche di performance sulle classi rare e comuni")
print("5️⃣ L'applicazione di tecniche di bilanciamento del dataset è essenziale per evitare bias verso le classi più rappresentate")
print("6️⃣ Per le classi con pochissimi esempi, tecniche come data augmentation specifica per audio (time shift, pitch shift, noise injection) sono fortemente consigliate")

## 3. Configurazione del Modello e Parametri

Definiamo i parametri di configurazione per il preprocessamento audio, la creazione dello spettrogramma Mel e l'addestramento della CNN.

In [ ]:
# I parametri principali sono già definiti nella classe Config
# Verifichiamo l'esistenza delle directory e creiamo quelle necessarie per l'output

def setup_output_directories():
    """
    Configura le directory per l'output del progetto.
    
    Returns:
        dict: Dictionary con i percorsi delle directory di output
    """
    # Directory principale di output
    output_dir = config.OUTPUT_DIR
    
    # Sotto-directory per diversi tipi di output
    dirs = {
        'checkpoints': os.path.join(output_dir, 'checkpoints'),
        'tensorboard': os.path.join(output_dir, 'tensorboard_logs'),
        'predictions': os.path.join(output_dir, 'predictions'),
        'submissions': os.path.join(output_dir, 'submissions'),
        'visualizations': os.path.join(output_dir, 'visualizations'),
    }
    
    # Crea tutte le directory
    for dir_name, dir_path in dirs.items():
        os.makedirs(dir_path, exist_ok=True)
        print(f"Directory '{dir_name}' creata/verificata in: {dir_path}")
    
    return dirs

# Configura le directory di output
output_dirs = setup_output_directories()

# Crea un file di log per tenere traccia dei risultati
log_file_path = os.path.join(config.OUTPUT_DIR, f"experiment_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

with open(log_file_path, 'w') as log_file:
    log_file.write(f"=== BirdClef Experiment Log ===\n")
    log_file.write(f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    log_file.write("Output directories:\n")
    for dir_name, dir_path in output_dirs.items():
        log_file.write(f"- {dir_name}: {dir_path}\n")

print(f"File di log creato in: {log_file_path}")

# Memorizziamo i parametri di configurazione principali per l'addestramento
print("\nParametri di configurazione principali:")
print(f"- Sample rate: {config.SR} Hz")
print(f"- Durata clip audio: {config.DURATION} secondi")
print(f"- Numero bande Mel: {config.N_MELS}")
print(f"- Dimensione FFT: {config.N_FFT}")
print(f"- Hop length: {config.HOP_LENGTH}")
print(f"- Device: {config.DEVICE}")
print(f"- Batch size: {config.BATCH_SIZE}")
print(f"- Epoche: {config.EPOCHS}")

Directory 'checkpoints' creata/verificata in: /kaggle/working/checkpoints
Directory 'tensorboard' creata/verificata in: /kaggle/working/tensorboard_logs
Directory 'predictions' creata/verificata in: /kaggle/working/predictions
Directory 'submissions' creata/verificata in: /kaggle/working/submissions
Directory 'visualizations' creata/verificata in: /kaggle/working/visualizations
File di log creato in: /kaggle/working/experiment_log_20250604_171558.txt

Parametri di configurazione principali:
- Sample rate: 32000 Hz
- Durata clip audio: 5 secondi
- Numero bande Mel: 128
- Dimensione FFT: 1024
- Hop length: 512
- Device: cpu
- Batch size: 96
- Epoche: 10


## 4. Caricamento e Preprocessing dei Dati

In questa sezione carichiamo i metadati dal file CSV di training, creiamo codifiche one-hot per le etichette delle specie e implementiamo funzioni per il caricamento e preprocessamento dei file audio.

### Normalizzazione

In [ ]:
# Crea una singola istanza della trasformazione MelSpectrogram da riutilizzare
mel_transform = T.MelSpectrogram(
    sample_rate=config.SR,
    n_fft=config.N_FFT,
    win_length=None,
    hop_length=config.HOP_LENGTH,
    f_min=config.FMIN,
    f_max=config.FMAX,
    n_mels=config.N_MELS,
    window_fn=torch.hann_window,
    power=config.POWER,
    normalized=False,
    onesided=True,
    norm="slaney",
    mel_scale="slaney"
)

# Funzione di conversione a dB e normalizzazione
def amplitude_to_db(spectrogram):
    """Converti spettrogramma in scala dB e normalizza tra 0-1"""
    # Converti in dB
    spectrogram_db = 10.0 * torch.log10(torch.clamp(spectrogram, min=1e-10))
    
    # Normalizza
    min_val = torch.min(spectrogram_db)
    max_val = torch.max(spectrogram_db)
    if max_val > min_val:
        return (spectrogram_db - min_val) / (max_val - min_val)
    else:
        return torch.zeros_like(spectrogram_db)

### Creazione nuovi esempi con data augmentations

In [ ]:
# Attiva/disattiva la generazione dei file augmentati
GENERATE_AUGMENTED_FILES = True  # Imposta a False per saltare questa fase

if GENERATE_AUGMENTED_FILES:    
    print("=== Generazione di File Audio Aumentati per Bilanciare il Dataset ===")
    
    class AudioAugmentation:
        @staticmethod
        def tshift(audio, sr):
            timeshift = Shift(min_shift=-0.25, max_shift=0.25, shift_unit="fraction", rollover=True, p=1)
            call = timeshift(audio, sr)
            effect = "tshift"
            return call, effect
    
        @staticmethod
        def gaussian_noise(audio, sr):
            augmenter = AddGaussianNoise(min_amplitude=0.01, max_amplitude=0.015, p=1)
            call = augmenter(audio, sr)
            effect = "gaussian_noise"
            return call, effect
    
        @staticmethod
        def ts_bandsf(audio, sr):
            # Imposta il massimo centro banda < Nyquist (sr/2)
            max_center_freq = sr // 2 - 1000  # margine di sicurezza 1kHz
            timeshift = Shift(min_shift=-0.25, max_shift=0.25, shift_unit="fraction", rollover=True, p=1)
            ts_call = timeshift(audio, sr)
            bandsf = BandStopFilter(
                min_center_freq=1000,
                max_center_freq=max_center_freq,
                min_bandwidth_fraction=0.1,
                max_bandwidth_fraction=0.5,
                p=1
            )
            call = bandsf(ts_call, sr)
            effect = "ts_bandsf"
            return call, effect
    
        @staticmethod
        def ts_tmask(audio, sr):
            timeshift = Shift(min_shift=-0.25, max_shift=0.25, shift_unit="fraction", rollover=True, p=1)
            ts_call = timeshift(audio, sr)
            tmask = TimeMask(min_band_part=0.02, max_band_part=0.05, fade_duration=0.0, p=1)
            call = tmask(ts_call, sr)
            effect = "ts_tmask"
            return call, effect
    
        @staticmethod
        def ts_bsf_gaussian(audio, sr):
            max_center_freq = sr // 2 - 1000  # margine di sicurezza 1kHz
            timeshift = Shift(min_shift=-0.25, max_shift=0.25, shift_unit="fraction", rollover=True, p=1)
            ts_call = timeshift(audio, sr)
            augmenter = AddGaussianNoise(min_amplitude=0.01, max_amplitude=0.015, p=1)
            ts_noise_call = augmenter(ts_call, sr)
            bandsf = BandStopFilter(
                min_center_freq=1000,
                max_center_freq=max_center_freq,
                min_bandwidth_fraction=0.1,
                max_bandwidth_fraction=0.5,
                p=1
            )
            call = bandsf(ts_noise_call, sr)
            effect = "ts_bsf_gaussian"
            return call, effect
    
        @staticmethod
        def ts_noise(audio, sr):
            timeshift = Shift(min_shift=-0.25, max_shift=0.25, shift_unit="fraction", rollover=True, p=1)
            ts_call = timeshift(audio, sr)
            augmenter = AddGaussianNoise(min_amplitude=0.01, max_amplitude=0.015, p=1)
            call = augmenter(ts_call, sr)
            effect = "ts_noise"
            return call, effect
    
    def generate_augmented_files(train_df, target_min_samples=40, aug_dir=None):
        """
        Genera file audio aumentati per bilanciare il dataset.
        
        Args:
            train_df: DataFrame con i metadati del training
            target_min_samples: Numero minimo di campioni desiderato per classe
            aug_dir: Directory dove salvare i file aumentati (default: config.TRAIN_AUDIO_DIR + '_aug')
        
        Returns:
            DataFrame aggiornato con i file aumentati inclusi
        """
        # Se non è specificata la directory, usa una sottodirectory della cartella audio
        if aug_dir is None:
            aug_dir = os.path.join("/kaggle/working", 'train_audio_aug')
        
        os.makedirs(aug_dir, exist_ok=True)
        print(f"Directory per i file aumentati: {aug_dir}")
        
        # Lista degli effetti di augmentation disponibili
        aug_effects = ['tshift', 'gaussian_noise', 'ts_bandsf', 'ts_tmask', 'ts_bsf_gaussian', 'ts_noise']
        
        # Conta i campioni per ogni specie
        species_counts = train_df['primary_label'].value_counts()
        
        # Ottieni le classi che necessitano di augmentation
        classes_needing_aug = species_counts[species_counts < target_min_samples].index.tolist()
        print(f"Trovate {len(classes_needing_aug)} classi con meno di {target_min_samples} esempi.")
        
        # Se non ci sono classi che necessitano di augmentation, restituisci il DataFrame originale
        if not classes_needing_aug:
            print("Tutte le classi hanno già il numero minimo di campioni. Nessuna augmentation necessaria.")
            return train_df
        
        # Lista per memorizzare le nuove righe del DataFrame
        new_rows = []
        total_augmented = 0
        
        # Processa ogni classe con pochi campioni
        for species in tqdm(classes_needing_aug, desc="Generazione file aumentati per classi rare"):
            # Filtra solo i file di questa specie
            species_files = train_df[train_df['primary_label'] == species]
            n_files = len(species_files)
            
            # Calcola quanti file augmentati creare
            n_needed = target_min_samples - n_files
            if n_needed <= 0:
                continue
                
            # Crea directory per la specie se non esiste
            species_dir = os.path.join(aug_dir, species)
            os.makedirs(species_dir, exist_ok=True)
            
            # Calcola quante augmentation per file originale
            rpt = math.ceil(n_needed / n_files)
            
            # Contatore dei file creati per questa specie
            files_created = 0
            
            # Processa ogni file della specie corrente
            for _, row in species_files.iterrows():
                file_path = os.path.join(config.TRAIN_AUDIO_DIR, row['filename'])
                base_filename = os.path.basename(file_path)
                fname = os.path.splitext(base_filename)[0]
                
                try:
                    # Carica l'audio
                    audio, sr = librosa.load(file_path, sr=config.SR)
                    
                    # Limita a 5 secondi se più lungo
                    if len(audio) > sr * 5:
                        audio = audio[:sr * 5]
                    
                    # Applica augmentation fino a raggiungere il target o il max repeat
                    for i in range(rpt):
                        if files_created >= n_needed:
                            break
                            
                        # Seleziona casualmente un effetto
                        effect_name = random.choice(aug_effects)
                        effect_func = getattr(AudioAugmentation, effect_name)
                        
                        # Applica l'effetto
                        aug_audio, effect_label = effect_func(audio, sr)
                        
                        # Crea nome file per l'audio aumentato
                        aug_filename = f"{fname}_{effect_label}_{i}.wav"
                        aug_filepath = os.path.join(species_dir, aug_filename)
                        
                        # Salva file aumentato
                        sf.write(aug_filepath, aug_audio, sr)
                        
                        # Crea riga per il DataFrame con le informazioni del file originale
                        new_row = row.copy()
                        new_row['filename'] = os.path.join(species, aug_filename)
                        new_row['is_augmented'] = 1
                        new_rows.append(new_row)
                        
                        files_created += 1
                        total_augmented += 1
                        
                except Exception as e:
                    print(f"Errore nell'elaborazione di {file_path}: {e}")
            
            print(f"- {species}: creati {files_created} file aumentati (totale: {n_files + files_created})")
        
        # Aggiungi le nuove righe al DataFrame originale
        augmented_df = pd.concat([train_df, pd.DataFrame(new_rows)], ignore_index=True)
        
        # Aggiungi la colonna is_augmented se non esiste
        if 'is_augmented' not in augmented_df.columns:
            augmented_df['is_augmented'] = 0
            
        print(f"\nTotale file aumentati creati: {total_augmented}")
        print(f"Dimensione DataFrame originale: {len(train_df)}")
        print(f"Dimensione DataFrame con file aumentati: {len(augmented_df)}")
        
        # Verifica le nuove statistiche
        new_counts = augmented_df['primary_label'].value_counts()
        still_below = new_counts[new_counts < target_min_samples]
        if len(still_below) > 0:
            print(f"Attenzione: {len(still_below)} classi hanno ancora meno di {target_min_samples} esempi.")
        else:
            print(f"Tutte le classi hanno ora almeno {target_min_samples} esempi!")
            
        return augmented_df
    
    # Carica i metadati originali se non già disponibili
    if 'train_df' not in globals():
        print("Caricamento dei metadati originali...")
        train_df, all_species, train_labels_one_hot, species_to_int, int_to_species = load_metadata()
    
    # Esegui la generazione dei file aumentati
    augmented_train_df = generate_augmented_files(train_df, target_min_samples=40)
    
    # Aggiorna il DataFrame di training e le etichette one-hot
    train_df = augmented_train_df
    
    # Ricalcola le etichette one-hot per includere i file aumentati
    mlb = MultiLabelBinarizer(classes=all_species)
    mlb.fit(None)
    
    train_df['all_labels'] = train_df.apply(
        lambda row: eval(row['secondary_labels']) + [row['primary_label']], 
        axis=1
    )
    train_labels_one_hot = mlb.transform(train_df['all_labels'])
    
    print("Dataset aggiornato con file aumentati!")
else:
    print("Generazione file aumentati disattivata.")

In [ ]:
# Suddividi i dati in training e validation
def split_data(train_df, labels_one_hot, test_size=0.2, random_state=42):
    """
    Suddivide il dataset in set di training e validation mantenendo la distribuzione delle classi.
    
    Args:
        train_df: DataFrame con i metadati
        labels_one_hot: Array di etichette one-hot
        test_size: Percentuale dei dati da usare per validation
        random_state: Seed per riproducibilità
        
    Returns:
        tuple: X_train_df, X_val_df, y_train_one_hot, y_val_one_hot
    """
    # Indici per lo split STRATIFICATO basato sulle etichette primarie
    train_indices, val_indices = train_test_split(
        range(len(train_df)),
        test_size=test_size,
        random_state=random_state,
        stratify=train_df['primary_label']  # Aggiunto parametro stratify
    )
    
    # Crea i dataframe e gli array di etichette splittati
    X_train_df = train_df.iloc[train_indices].reset_index(drop=True)
    X_val_df = train_df.iloc[val_indices].reset_index(drop=True)
    
    y_train_one_hot = labels_one_hot[train_indices]
    y_val_one_hot = labels_one_hot[val_indices]
    
    print(f"Dimensioni Training Set: {X_train_df.shape}, Etichette: {y_train_one_hot.shape}")
    print(f"Dimensioni Validation Set: {X_val_df.shape}, Etichette: {y_val_one_hot.shape}")
    
    # Verifica presenza di tutte le classi nei set
    train_classes = set(X_train_df['primary_label'].unique())
    val_classes = set(X_val_df['primary_label'].unique())
    all_dataset_classes = set(train_df['primary_label'].unique())
    
    print(f"Classi totali nel dataset: {len(all_dataset_classes)}")
    print(f"Classi nel training set: {len(train_classes)}")
    print(f"Classi nel validation set: {len(val_classes)}")
    
    # Verifica classi mancanti
    missing_in_train = all_dataset_classes - train_classes
    missing_in_val = all_dataset_classes - val_classes
    
    if missing_in_train:
        print(f"ATTENZIONE: {len(missing_in_train)} classi mancanti nel training set!")
    if missing_in_val:
        print(f"ATTENZIONE: {len(missing_in_val)} classi mancanti nel validation set!")
    
    return X_train_df, X_val_df, y_train_one_hot, y_val_one_hot

# Ora procedi con lo split train/val stratificato
X_train_df, X_val_df, y_train_one_hot, y_val_one_hot = split_data(train_df, train_labels_one_hot)
    
# Per Kaggle, dovremo creare un dataset speciale per le soundscapes di test
# Questo verrà utilizzato direttamente nella fase di generazione della submission
# Non creiamo X_test_df e test_dataset per ora
X_test_df = None
y_test_one_hot = None

## Funzione di bilanciamento del dataset - Cancella una percentuale di esempi dalle classi molto numerose

In [9]:
def create_balanced_dataset_df(train_df, labels_one_hot, abundant_class_threshold=200, remove_percentage=0.3, random_state=42):
    """
    Crea un DataFrame bilanciato rimuovendo parte degli esempi con rating bassi dalle classi abbondanti.
    
    Args:
        train_df: DataFrame originale
        labels_one_hot: Array di etichette one-hot
        abundant_class_threshold: Soglia per definire una classe come "abbondante"
        remove_percentage: Percentuale di esempi con rating 1-3 da rimuovere dalle classi abbondanti
        random_state: Seed per riproducibilità
        
    Returns:
        tuple: (DataFrame bilanciato, etichette one-hot bilanciate)
    """
    # Conta esempi per ogni classe
    class_counts = train_df['primary_label'].value_counts()
    
    # Identifica classi abbondanti
    abundant_classes = class_counts[class_counts > abundant_class_threshold].index.tolist()
    print(f"Classi identificate come abbondanti (>{abundant_class_threshold} esempi): {len(abundant_classes)}")
    
    # Copia il DataFrame originale
    balanced_df = train_df.copy()
    rows_to_drop = []
    
    # Contatori per statistiche
    total_removed = 0
    removed_by_class = {}
    
    # Per ogni classe abbondante
    for cls in abundant_classes:
        # Filtra esempi con rating 1-3 per questa classe
        low_quality_mask = (balanced_df['primary_label'] == cls) & (balanced_df['rating'].isin([1, 2, 3]))
        low_quality_indices = balanced_df[low_quality_mask].index.tolist()
        
        # Numero di esempi da rimuovere
        n_to_remove = int(len(low_quality_indices) * remove_percentage)
        
        # Seleziona casualmente gli indici da rimuovere
        np.random.seed(random_state)
        if n_to_remove > 0:
            indices_to_remove = np.random.choice(low_quality_indices, size=n_to_remove, replace=False)
            
            # Memorizza gli indici da rimuovere
            rows_to_drop.extend(indices_to_remove)
            
            # Aggiorna statistiche
            removed_by_class[cls] = n_to_remove
            total_removed += n_to_remove
    
    # Rimuovi le righe selezionate
    if rows_to_drop:
        balanced_df = balanced_df.drop(rows_to_drop).reset_index(drop=True)
        
        # Aggiorna anche le etichette one-hot rimuovendo gli stessi indici
        mask = np.ones(len(train_df), dtype=bool)
        mask[rows_to_drop] = False
        balanced_labels = labels_one_hot[mask]
    else:
        balanced_labels = labels_one_hot
    
    # Statistiche finali
    print(f"Totale esempi rimossi: {total_removed} ({total_removed/len(train_df):.1%} del dataset originale)")
    print(f"Dimensione dataset originale: {len(train_df)}")
    print(f"Dimensione dataset bilanciato: {len(balanced_df)}")
    
    # Visualizza le prime 5 classi con maggiori rimozioni
    if removed_by_class:
        top_removed = sorted(removed_by_class.items(), key=lambda x: x[1], reverse=True)[:5]
        print("\nClassi con maggior numero di esempi rimossi:")
        for cls, count in top_removed:
            original = class_counts[cls]
            remaining = original - count
            print(f"- {cls}: {count} rimossi, {remaining}/{original} rimanenti ({remaining/original:.1%})")
    else:
        print("Nessun esempio rimosso.")
    
    return balanced_df, balanced_labels

## Data Augmentation: Implementazione delle Tecniche dei Vincitori

In questa sezione implementiamo le tre tecniche di data augmentation che hanno contribuito significativamente alle performance dei vincitori:
1. **Random Segment Selection** - Estrae segmenti casuali dalle registrazioni audio
2. **XY Masking** - Applica maschere casuali sugli assi tempo e frequenza degli spettrogrammi Mel
3. **Horizontal CutMix** - Combina parti di spettrogrammi da diverse registrazioni

La classe `AudioAugmentations` gestisce tutte queste trasformazioni in modo unificato.

In [11]:
class AudioAugmentations:
    def __init__(self, p_random_segment=0.5, p_xy_mask=0.5, p_horizontal_cutmix=0.25):
        """
        Inizializza le trasformazioni per data augmentation audio.
        
        Args:
            p_random_segment: Probabilità di utilizzare un segmento casuale
            p_xy_mask: Probabilità di applicare il mascheramento XY
            p_horizontal_cutmix: Probabilità di applicare horizontal cutmix
        """
        self.p_random_segment = p_random_segment
        self.p_xy_mask = p_xy_mask
        self.p_horizontal_cutmix = p_horizontal_cutmix
    
    def apply_xy_masking(self, spec):
        """Applica maschere casuali sull'asse X (tempo) e Y (frequenza) allo spettrogramma"""
        mask = spec.clone()
        
        # Determina la dimensionalità del tensore
        if len(mask.shape) == 3:  # [channels, height, width]
            channels, height, width = mask.shape
        elif len(mask.shape) == 4:  # [batch, channels, height, width]
            _, channels, height, width = mask.shape
        else:
            raise ValueError(f"Forma dello spettrogramma non supportata: {mask.shape}")
        
        # Masking temporale (asse X)
        if np.random.random() < self.p_xy_mask:
            mask_width = int(width * np.random.uniform(0.1, 0.2))  # 10-20% width
            mask_start = np.random.randint(0, width - mask_width)
            mask[..., mask_start:mask_start+mask_width] = 0
        
        # Masking frequenziale (asse Y)
        if np.random.random() < self.p_xy_mask:
            mask_height = int(height * np.random.uniform(0.1, 0.2))  # 10-20% height
            mask_start = np.random.randint(0, height - mask_height)
            
            # Adatta l'indicizzazione in base alla dimensionalità
            if len(mask.shape) == 3:  # [channels, height, width]
                mask[:, mask_start:mask_start+mask_height, :] = 0
            else:  # [batch, channels, height, width]
                mask[:, :, mask_start:mask_start+mask_height, :] = 0
            
        return mask

### Horizontal CutMix: Implementazione della Funzione di Collate

Questa funzione personalizzata viene utilizzata nel DataLoader per implementare l'Horizontal CutMix,
che combina sezioni temporali di spettrogrammi diversi all'interno dello stesso batch.
Le etichette vengono miscelate proporzionalmente alla quantità di dati combinati.

In [12]:
def horizontal_cutmix_collate(batch, p_cutmix=0.25):
    """
    Collate function che applica horizontal cutmix tra elementi del batch con probabilità p_cutmix.
    
    Args:
        batch: Lista di tuple (input, target)
        p_cutmix: Probabilità di applicare cutmix ad ogni coppia di esempi
    
    Returns:
        tuple: (inputs_batch, targets_batch)
    """
    inputs = []
    targets = []
    
    # Estrae input e target dal batch
    for input_tensor, target_tensor in batch:
        inputs.append(input_tensor)
        targets.append(target_tensor)
    
    inputs = torch.stack(inputs)
    targets = torch.stack(targets)
    batch_size = len(inputs)
    
    # Applica horizontal cutmix
    if batch_size > 1:  # Serve almeno 2 elementi per fare cutmix
        for i in range(batch_size):
            if np.random.rand() < p_cutmix:
                # Seleziona un altro esempio casuale da mixare
                j = np.random.randint(0, batch_size)
                if i != j:
                    # Determina il punto di taglio orizzontale (sulla dimensione temporale)
                    width = inputs.shape[3]
                    cut_point = np.random.randint(int(width * 0.25), int(width * 0.75))
                    
                    # Esegue il mixing
                    mix_ratio = cut_point / width
                    inputs[i, :, :, :cut_point] = inputs[j, :, :, :cut_point]
                    
                    # Mix delle etichette in proporzione al mix degli input
                    targets[i] = targets[i] * (1 - mix_ratio) + targets[j] * mix_ratio
    
    return inputs, targets

## Caricamento e pre-processing

## Preprocessing Audio con Supporto per Data Augmentation

La funzione di caricamento audio è stata modificata per supportare due modalità di estrazione:
- **Posizionale** - Estrazione da punti specifici nella registrazione ('start', 'center', 'end')
- **Casuale** - Estrazione di un segmento casuale quando `random_segment=True`

Questa implementazione permette di applicare la tecnica di Random Segment Selection come parte della data augmentation.

In [ ]:
def load_and_preprocess_audio_torch(file_path, target_sr=config.SR, duration=config.DURATION, 
                                    segment_position='center', random_segment=False):
    """
    Carica un file audio, estrae un segmento specifico e lo converte in spettrogramma Mel usando PyTorch.
    """
    try:
        # Carica il file audio con torchaudio
        waveform, sr = torchaudio.load(file_path)
        
        # Converti a mono se necessario (prendi il primo canale se stereo)
        if waveform.shape[0] > 1:
            waveform = waveform[0:1]
        
        # Ricampiona se necessario
        if sr != target_sr:
            resampler = T.Resample(sr, target_sr)
            waveform = resampler(waveform)
        
        # Calcola la lunghezza target in campioni
        target_len = int(target_sr * duration)
        total_len = waveform.shape[1]
        
        # Gestisci clip troppo corte
        if total_len < target_len:
            # Ripeti l'audio per raggiungere la lunghezza target
            n_repeat = (target_len // total_len) + 1
            waveform = waveform.repeat(1, n_repeat)
            total_len = waveform.shape[1]
        
        # Seleziona il segmento
        if random_segment and total_len > target_len:
            # Estrai segmento casuale
            max_start_idx = total_len - target_len
            start_idx = torch.randint(0, max_start_idx, (1,)).item()
        else:
            # Usa le posizioni predefinite
            if segment_position == 'start':
                start_idx = int(total_len * 0.2)
                if start_idx + target_len > total_len:
                    start_idx = max(0, total_len - target_len)
            elif segment_position == 'end':
                end_point = int(total_len * 0.8)
                start_idx = max(0, end_point - target_len)
            else:  # 'center' (default)
                start_idx = max(0, int(total_len / 2 - target_len / 2))
        
        # Estrai il segmento
        waveform = waveform[:, start_idx:start_idx + target_len]
        
        # Padda se necessario
        if waveform.shape[1] < target_len:
            padding = target_len - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        
        # Applica la trasformazione Mel
        mel_spectrogram = mel_transform(waveform)
        
        # Converti in scala dB e normalizza
        log_mel_spec = amplitude_to_db(mel_spectrogram)

        resize_transform = transforms.Resize((224, 224), 
            interpolation=transforms.InterpolationMode.BICUBIC)
        log_mel_spec = resize_transform(log_mel_spec)
        
        return log_mel_spec
        
    except Exception as e:
        print(f"Errore nell'elaborazione di {file_path}: {e}")
        # Crea uno spettrogramma vuoto
        time_steps = int(target_sr * duration / config.HOP_LENGTH) + 1
        return torch.zeros((1, config.N_MELS, time_steps), dtype=torch.float32)

## 5. Dataset PyTorch per Dati Audio

## Dataset PyTorch con Supporto Integrato per Data Augmentation

### BirdDataset con l'utilizzo di adaptive clip in base alla lunghezza

In [ ]:
class SimpleBirdDataset(Dataset):
    def __init__(self, df, audio_dir, labels_one_hot, transform=None, augmentations=None, aug_dir=None):
        """Dataset che estrae un singolo segmento casuale da ogni clip audio."""
        self.df = df
        self.audio_dir = audio_dir
        self.aug_dir = aug_dir
        self.labels = labels_one_hot
        self.transform = transform
        self.augmentations = augmentations
    
    def _get_file_path(self, filename):
        """
        Cerca il file prima nella directory dei file aumentati, poi nella directory originale.
        """
        if self.aug_dir is not None:
            aug_path = os.path.join(self.aug_dir, filename)
            if os.path.exists(aug_path):
                return aug_path
        return os.path.join(self.audio_dir, filename)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Ottieni il record dal DataFrame
        row = self.df.iloc[idx]
        filename = row['filename']
        
        # Trova il percorso del file
        file_path = self._get_file_path(filename)
        
        # Carica e preprocessa l'audio con estrazione sempre casuale
        mel_spec_tensor = load_and_preprocess_audio_torch(
            file_path,
            random_segment=True  # Sempre casuale
        )
        
        # Applica le augmentations come prima
        if self.augmentations and np.random.random() < self.augmentations.p_xy_mask:
            mel_spec_tensor = self.augmentations.apply_xy_masking(mel_spec_tensor)
        
        if self.transform:
            mel_spec_tensor = self.transform(mel_spec_tensor)
            
        # Ottieni le etichette
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.float32)
            
        return mel_spec_tensor, label_tensor

### Creazione dataset e dataloader con AdaptiveMultiSegment

### Creazione dei DataLoader con Augmentation

In questa sezione:
1. Applichiamo il bilanciamento strategico al dataset di training
2. Creiamo l'istanza di AudioAugmentations con le probabilità ottimali
3. Configuriamo i dataloader con le funzioni di collate personalizzate
4. Analizziamo la distribuzione dei segmenti nel dataset risultante

In [ ]:
# Applica il bilanciamento strategico solo al dataset di training
print("\n=== Bilanciamento Strategico del Dataset di Training ===")
X_train_df_balanced, y_train_one_hot_balanced = create_balanced_dataset_df(
    X_train_df, 
    y_train_one_hot,
    abundant_class_threshold=180,  # Classi con più di 180 esempi sono considerate abbondanti
    remove_percentage=0.5  # Rimuove il 50% degli esempi con rating bassi
)

# Ottieni la directory dei file aumentati
aug_dir = os.path.join("/kaggle/working", 'train_audio_aug')

# Dataset di training con entrambe le directory
train_dataset = SimpleBirdDataset(
    X_train_df_balanced, 
    config.TRAIN_AUDIO_DIR, 
    y_train_one_hot_balanced,
    augmentations=AudioAugmentations(p_random_segment=1.0, p_xy_mask=0.7, p_horizontal_cutmix=0.4),
    aug_dir=aug_dir  # Passa la directory dei file aumentati
)
print("Creazione dataset di validation...")
# Per validation, non usiamo augmentation
val_dataset = SimpleBirdDataset(
    X_val_df, 
    config.TRAIN_AUDIO_DIR, 
    y_val_one_hot,
    augmentations=AudioAugmentations(p_random_segment=1.0, p_xy_mask=0.0, p_horizontal_cutmix=0.0),
    aug_dir=aug_dir  # Passa la directory dei file aumentati
)

# Stampa informazioni sulla dimensione effettiva del dataset
print(f"\nNumero di record originali nel training set: {len(X_train_df)}")
print(f"Numero di campioni effettivi nel training set dopo l'adattamento: {len(train_dataset)}")

# Creiamo i dataloader con horizontal cutmix per il training
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=True,
    num_workers=config.NUM_WORKERS, 
    pin_memory=True,
    persistent_workers=True,  # Mantiene i worker attivi tra le epoche
    prefetch_factor=4,        # Aumenta il prefetching
    collate_fn=lambda batch: horizontal_cutmix_collate(batch, p_cutmix=0.4)
)

# Sostituisci val_loader con questa versione
val_loader = DataLoader(
    val_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False,
    num_workers=config.NUM_WORKERS, 
    pin_memory=True,
    persistent_workers=True   # Mantiene i worker attivi tra le epoche
)

# Non creiamo un test_loader per ora
test_loader = None
    
print(f"Numero di batch di training per epoca: {len(train_loader)}")
print(f"Numero di batch di validation per epoca: {len(val_loader)}")
print("Test set: utilizzerò direttamente i file nella cartella test_soundscapes")


=== Bilanciamento Strategico del Dataset di Training ===
Classi identificate come abbondanti (>150 esempi): 49
Totale esempi rimossi: 707 (3.1% del dataset originale)
Dimensione dataset originale: 22851
Dimensione dataset bilanciato: 22144

Classi con maggior numero di esempi rimossi:
- roahaw: 34 rimossi, 551/585 rimanenti (94.2%)
- banana: 26 rimossi, 471/497 rimanenti (94.8%)
- grekis: 25 rimossi, 762/787 rimanenti (96.8%)
- compau: 25 rimossi, 607/632 rimanenti (96.0%)
- littin1: 23 rimossi, 247/270 rimanenti (91.5%)
Creazione dataset di training con approccio multi-segmento adattivo...
Analizzando le lunghezze delle clip audio...


Preparazione dataset adattivo:   0%|          | 0/22144 [00:00<?, ?it/s]

Creazione dataset di validation con segmento centrale...

Numero di record originali nel training set: 22851
Numero di campioni effettivi nel training set dopo l'adattamento: 56330
Rapporto di espansione: 2.47x

Distribuzione dei segmenti nel dataset:
- start: 18764 (33.3%)
- center: 18802 (33.4%)
- end: 18764 (33.3%)
Numero di batch di training per epoca: 587
Numero di batch di validation per epoca: 60
Test set: utilizzerò direttamente i file nella cartella test_soundscapes


## 6. Definizione del Modello CNN

## Modello EfficientNet con Head Personalizzata

Implementiamo un modello basato su EfficientNet-B0 preaddestrato, con:
- Supporto per input a singolo canale (spettrogrammi Mel)
- Testa di classificazione personalizzata con dropout e normalizzazione batch
- Parametri differenziati per l'ottimizzazione
- Gestione automatica dei checkpoint e dei pesi preaddestrati

In [ ]:
class EfficientNetBirdClassifier(nn.Module):
    def __init__(self, num_classes=config.N_CLASSES, pretrained=True, model_name='tf_efficientnet_b0.ns_jft_in1k'):
        super(EfficientNetBirdClassifier, self).__init__()
        
        # Tenta di caricare il modello con pesi pre-addestrati, gestendo fallimenti di connessione
        try:
            if pretrained:
                print(f"Tentativo di caricare {model_name} con pesi pre-addestrati...")
                self.efficientnet = timm.create_model(
                    model_name,
                    pretrained=True,
                    num_classes=0  # Rimuovi il classificatore originale
                )
                print("Modello caricato con successo con pesi pre-addestrati.")
            else:
                # Se pretrained=False, non tentare di scaricare i pesi
                self.efficientnet = timm.create_model(
                    model_name,
                    pretrained=False,  # CORRETTO: ora è False
                    num_classes=0  # Rimuovi il classificatore originale
                )
                print("Modello inizializzato senza pesi pre-addestrati (modalità offline).")
        except Exception as e:
            print(f"Errore nel caricamento dei pesi pre-addestrati: {e}")
            print("Inizializzazione del modello senza pesi pre-addestrati...")
            self.efficientnet = timm.create_model(
                model_name,
                pretrained=False,
                num_classes=0  # Rimuovi il classificatore originale
            )
        
        # Ottieni la dimensione dell'output del feature extractor
        if hasattr(self.efficientnet, 'num_features'):
            classifier_in_features = self.efficientnet.num_features
        elif hasattr(self.efficientnet, 'classifier'):
            classifier_in_features = self.efficientnet.classifier.in_features
        else:
            # Valore predefinito per EfficientNet-B0
            classifier_in_features = 1280
        
         # Sostituisci il classificatore semplice con una MLP con dropout
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),  # Primo dropout significativo
            nn.Linear(classifier_in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),  # Secondo dropout più leggero
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        # Se l'input è un'immagine a 1 canale, replicala su 3 canali
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        
        # Passa l'input attraverso il backbone per ottenere le features
        features = self.efficientnet(x)
        
        # Passa le feature attraverso il classificatore
        output = self.classifier(features)
        
        return output

# Controlla se esiste un checkpoint precedente
has_previous_checkpoint = False
latest_checkpoint = '/kaggle/working/checkpoints/latest_checkpoint.pth'
has_previous_checkpoint = os.path.exists(latest_checkpoint)

# Inizializza il modello - usa pretrained=False se hai già un checkpoint
model = EfficientNetBirdClassifier(
    num_classes=config.N_CLASSES, 
    pretrained=not has_previous_checkpoint  # Scarica i pesi solo se non c'è già un checkpoint
).to(config.DEVICE)

# Definiamo ottimizzatore con learning rate differenziati
def get_optimizer(model, lr_base=4e-4, lr_head=8e-4):  # Ridotti significativamente
    # Parametri del corpo (pre-addestrati)
    backbone_params = [p for name, p in model.named_parameters() 
                      if 'classifier' not in name]
    
    # Parametri della testa (da addestrare da zero)
    classifier_params = [p for name, p in model.named_parameters() 
                      if 'classifier' in name]
    
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': lr_base},  # Ridotto da 1e-3 a 5e-4
        {'params': classifier_params, 'lr': lr_head}  # Ridotto da 3e-3 a 1e-3
    ], weight_decay=5e-4)
    
    return optimizer

# Loss function - manteniamo BCE come richiesto
criterion = nn.BCEWithLogitsLoss()  

# Ottimizzatore con learning rate differenziati
optimizer = get_optimizer(model)

# Learning rate scheduler - aggiornato a CosineAnnealingLR come usato dai vincitori
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.EPOCHS * 1.5,  # Numero totale di epoche
    eta_min=5e-7  # Learning rate minimo
)

print(f"Modello EfficientNet-B0 (timm) caricato su {config.DEVICE}")
print(f"Numero di classi: {config.N_CLASSES}")
print(f"Parametri totali: {sum(p.numel() for p in model.parameters()):,}")

Tentativo di caricare efficientnet_b0 con pesi pre-addestrati...
Errore nel caricamento dei pesi pre-addestrati: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.
Inizializzazione del modello senza pesi pre-addestrati...
Modello EfficientNet-B0 (timm) caricato su cpu
Numero di classi: 206
Parametri totali: 4,770,122


## 7. Addestramento e Validazione del Modello

## Funzione di Training con Supporto per Checkpoint

La funzione di training implementa:
- Caricamento automatico dei checkpoint precedenti
- Early stopping basato sulle performance di validation
- Salvataggio periodico dei checkpoint e del miglior modello
- Supporto per scheduler di learning rate (CosineAnnealingLR)
- Visualizzazione delle curve di loss

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, 
                epochs=config.EPOCHS, device=config.DEVICE, 
                model_save_path=None, model_load_path=None, patience=5,
                resume_training=True, scheduler=None):
    """
    Addestra il modello e valuta su validation set con supporto per checkpoint.
    
    Args:
        model: Modello PyTorch da addestrare
        train_loader: DataLoader per dati di training
        val_loader: DataLoader per dati di validation
        criterion: Funzione di loss
        optimizer: Ottimizzatore
        epochs: Numero di epoche di training
        device: Device per l'addestramento ('cuda' o 'cpu')
        model_save_path: Path dove salvare il modello addestrato
        model_load_path: Path da cui caricare un modello pre-addestrato
        patience: Numero di epoche senza miglioramento prima di terminare l'addestramento
        resume_training: Se True, riprende il training da un checkpoint (se disponibile)
        scheduler: Learning rate scheduler
        
    Returns:
        tuple: (train_losses, val_losses, total_training_time)
    """
    # Directory per i checkpoint in base all'ambiente
    checkpoint_dir = None
    
    # In Kaggle, usa la directory di working
    checkpoint_dir = '/kaggle/working/checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"Directory per i checkpoint creata in Kaggle: {checkpoint_dir}")
    
    # Inizializzazione variabili
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    total_training_time = 0
    start_epoch = 0
    needs_training = True
    checkpoint_exists = False
    model_loaded = False
    
    # Verifica se esiste un modello pre-addestrato da caricare
    if model_load_path and os.path.exists(model_load_path):
        print(f"Modello trovato in {model_load_path}. Tentativo di caricamento...")
        try:
            checkpoint = torch.load(model_load_path, map_location=device)
            
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
                
            print("Modello caricato con successo.")
            model_loaded = True
            needs_training = False
        except Exception as e:
            print(f"Errore durante il caricamento del modello: {e}")
            print("Verrà avviato l'addestramento da zero.")
            needs_training = True
    else:
        print(f"Modello non trovato in {model_load_path}.")
    
    # Cerca un checkpoint SOLO se il caricamento del modello è fallito E resume_training è True
    if needs_training and resume_training and checkpoint_dir and not model_loaded:
        latest_checkpoint = os.path.join(checkpoint_dir, "latest_checkpoint.pth")
        if os.path.exists(latest_checkpoint):
            print(f"Trovato checkpoint in {latest_checkpoint}. Tentativo di caricamento...")
            try:
                checkpoint = torch.load(latest_checkpoint, map_location=device)
                
                # Verifica che sia un checkpoint compatibile prima di caricarlo
                if isinstance(checkpoint, dict) and 'epoch' in checkpoint:
                    try:
                        model.load_state_dict(checkpoint['model_state_dict'])
                        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                        start_epoch = checkpoint['epoch'] + 1
                        train_losses = checkpoint['train_losses']
                        val_losses = checkpoint['val_losses']
                        best_val_loss = checkpoint['best_val_loss']
                        epochs_without_improvement = checkpoint['epochs_without_improvement']
                        total_training_time = checkpoint.get('total_training_time', 0)
                        
                        # Ricrea lo scheduler con lo stato salvato se presente
                        if scheduler is not None and 'scheduler_state_dict' in checkpoint:
                            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                        
                        print(f"Checkpoint caricato con successo (epoca {start_epoch-1})")
                        print(f"Si riparte dall'epoca {start_epoch}/{epochs}")
                        
                        if start_epoch >= epochs:
                            needs_training = False
                        
                        checkpoint_exists = True
                    except Exception as e:
                        print(f"Il checkpoint non è compatibile con il modello attuale: {e}")
                        print("Verrà avviato l'addestramento da zero.")
            except Exception as e:
                print(f"Errore durante il caricamento del checkpoint: {e}")
                print("Si procederà con il training da zero.")
    
    model.to(device)
    
    # Esegui training solo se necessario
    if needs_training:
        start_time_total = time.time()
        model.train()
        
        # Loop di training sulle epoche (inizia da start_epoch)
        for epoch in range(start_epoch, epochs):
            epoch_start_time = time.time()
            
            # --- Fase di Training ---
            model.train()
            running_loss = 0.0
            pbar_train = tqdm(enumerate(train_loader), total=len(train_loader), 
                             desc=f"Epoca {epoch+1}/{epochs} [Train]", leave=True)
            
            for i, (inputs, labels) in pbar_train:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
                running_loss += loss.item()
                avg_loss = running_loss / (i + 1)
                pbar_train.set_postfix({'loss': f"{avg_loss:.4f}"})
            
            epoch_train_loss = running_loss / len(train_loader)
            train_losses.append(epoch_train_loss)
            
            # --- Fase di Validation ---
            model.eval()
            running_val_loss = 0.0
            pbar_val = tqdm(enumerate(val_loader), total=len(val_loader), 
                           desc=f"Epoca {epoch+1}/{epochs} [Val]", leave=True)
            
            with torch.no_grad():
                for i, (val_inputs, val_labels) in pbar_val:
                    val_inputs = val_inputs.to(device)
                    val_labels = val_labels.to(device)
                    
                    val_outputs = model(val_inputs)
                    val_loss = criterion(val_outputs, val_labels)
                    running_val_loss += val_loss.item()
                    avg_val_loss = running_val_loss / (i + 1)
                    pbar_val.set_postfix({'val_loss': f"{avg_val_loss:.4f}"})
            
            epoch_val_loss = running_val_loss / len(val_loader)
            val_losses.append(epoch_val_loss)
            
            # Aggiornamento scheduler - modificato per CosineAnnealingLR
            if scheduler is not None:
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(epoch_val_loss)  # Per ReduceLROnPlateau
                else:
                    scheduler.step()  # Per CosineAnnealingLR è step() senza parametri
            
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - epoch_start_time
            total_training_time += epoch_duration
            
            print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {epoch_train_loss:.4f}, "
                  f"Val Loss: {epoch_val_loss:.4f}, Duration: {epoch_duration:.2f} sec")
            
            # Salvataggio checkpoint per ogni epoca (in qualsiasi ambiente)
            if checkpoint_dir:
                checkpoint_path = os.path.join(checkpoint_dir, f"birdclef_epoch_{epoch+1}.pth")
                
                # Salva checkpoint completo con tutte le informazioni di stato
                checkpoint = {
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_losses': train_losses,
                    'val_losses': val_losses,
                    'best_val_loss': best_val_loss,
                    'epochs_without_improvement': epochs_without_improvement,
                    'total_training_time': total_training_time
                }
                
                # Salva anche lo stato dello scheduler se esiste
                if scheduler is not None:
                    checkpoint['scheduler_state_dict'] = scheduler.state_dict()
                
                torch.save(checkpoint, checkpoint_path)
                print(f"Checkpoint completo salvato in {checkpoint_path}")
                
                # Aggiorna anche il checkpoint più recente (sovrascrive)
                torch.save(checkpoint, os.path.join(checkpoint_dir, "latest_checkpoint.pth"))
            
            # Early stopping
            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                epochs_without_improvement = 0
                # Salva il miglior modello separatamente
                if model_save_path:
                    best_path = model_save_path.replace('.pth', '_best.pth')
                    
                    # Salva checkpoint completo
                    best_checkpoint = {
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'train_losses': train_losses,
                        'val_losses': val_losses,
                        'best_val_loss': best_val_loss
                    }
                    
                    # Salva anche lo stato dello scheduler
                    if scheduler is not None:
                        best_checkpoint['scheduler_state_dict'] = scheduler.state_dict()
                    
                    torch.save(best_checkpoint, best_path)
                    print(f"Salvato miglior modello in {best_path}")
            else:
                epochs_without_improvement += 1
                
            if epochs_without_improvement >= patience:
                print(f"\nEarly stopping attivato! Nessun miglioramento per {patience} epoche consecutive.")
                break
        
        end_time_total = time.time()
        if checkpoint_exists:
            total_training_time += (end_time_total - start_time_total)
        else:
            total_training_time = end_time_total - start_time_total
            
        print(f"\nTraining terminato in {total_training_time/60:.2f} minuti totali")
        
        # Salva il modello finale
        if model_save_path:
            final_checkpoint = {
                'epoch': epochs-1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_losses': train_losses,
                'val_losses': val_losses,
                'best_val_loss': best_val_loss,
                'total_training_time': total_training_time
            }
            
            # Salva anche lo stato dello scheduler
            if scheduler is not None:
                final_checkpoint['scheduler_state_dict'] = scheduler.state_dict()
                
            torch.save(final_checkpoint, model_save_path)
            print(f"Modello finale salvato in {model_save_path}")
    else:
        print("Training non necessario: modello già caricato o training ripreso e completato.")
    
    # Visualizza le curve di loss
    if train_losses and val_losses:
        plt.figure(figsize=(10, 5))
        plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss')
        plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss')
        plt.xlabel('Epoche')
        plt.ylabel('Loss')
        plt.title('Curve di Loss di Training e Validation')
        plt.legend()
        plt.grid(True)
        plt.show()
        
        # Salva il grafico
        if checkpoint_dir:
            plt_path = os.path.join(checkpoint_dir, 'loss_curves.png')
            plt.savefig(plt_path)
            print(f"Grafico delle curve di loss salvato in {plt_path}")
    
    return train_losses, val_losses, total_training_time

### Configurazione e Avvio del Training

Configuriamo e avviamo il training del modello:
- Individuazione automatica dei checkpoint precedenti
- Inizializzazione dell'ottimizzatore e dello scheduler
- Avvio del training con i parametri ottimizzati

In [ ]:
# Directory per i checkpoint in Kaggle
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    
# Verifica se esiste un checkpoint precedente
latest_checkpoint = '/kaggle/working/checkpoints/latest_checkpoint.pth'
if os.path.exists(latest_checkpoint):
    model_load_path = latest_checkpoint
    print(f"Trovato checkpoint precedente in {latest_checkpoint}")
else:
    # Usa un modello base precaricato se disponibile
    model_load_path = "/kaggle/input/efficientnetb0_paramdiversi/pytorch/default/1/birdclef_efficientNET_refactor_nuoviParam_timm_best.pth"
        
    # Imposta il percorso di salvataggio
    model_save_path = "/kaggle/working/birdclef_efficientNET_refactor_nuoviParam_timm.pth"
    
# Addestra il modello con CosineAnnealingLR scheduler
train_losses, val_losses, training_time = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,  # Passa lo scheduler
    epochs=config.EPOCHS,
    device=config.DEVICE,
    model_save_path=model_save_path,
    model_load_path=model_load_path,
    resume_training=True
)

Directory per i checkpoint creata in Kaggle: /kaggle/working/checkpoints
Modello trovato in /kaggle/input/efficientnetb0_paramdiversi/pytorch/default/1/birdclef_efficientNET_refactor_nuoviParam_timm_best.pth. Tentativo di caricamento...
Modello caricato con successo.
Training non necessario: modello già caricato o training ripreso e completato.


### Metriche predictions

In [ ]:
def evaluate_model_on_validation(model, val_loader, device):
    """
    Valuta il modello sul validation set corrente, suddividendo le classi in categorie
    di frequenza basate sul numero di esempi presenti nel validation set stesso.
    
    Args:
        model: Modello addestrato da valutare
        val_loader: DataLoader per il validation set
        device: Device per l'inferenza
    """
    from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
    import matplotlib.pyplot as plt
    import numpy as np
    
    model.eval()
    
    # Inizializzazione array per raccogliere predizioni e target
    all_targets = []
    all_preds_prob = []
    all_preds_binary = []
    
    # Forward pass sul validation set
    with torch.no_grad():
        for inputs, targets in tqdm(val_loader, desc="Valutazione sul validation set"):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Forward pass
            outputs = model(inputs)
            probs = torch.sigmoid(outputs)
            
            # Soglia per la classificazione binaria
            binary_preds = (probs > 0.5).float()
            
            # Memorizza risultati
            all_targets.extend(targets.cpu().numpy())
            all_preds_prob.extend(probs.cpu().numpy())
            all_preds_binary.extend(binary_preds.cpu().numpy())
    
    # Converti in numpy array
    y_true = np.array(all_targets)
    y_pred_prob = np.array(all_preds_prob)
    y_pred = np.array(all_preds_binary)
    
    # Calcola la distribuzione delle classi nel validation set
    val_class_distribution = np.sum(y_true, axis=0)
    
    # Definisci soglie in base ai quartili della distribuzione
    threshold_rare = np.percentile(val_class_distribution[val_class_distribution > 0], 25)  # Primo quartile
    threshold_common = np.percentile(val_class_distribution, 75)  # Terzo quartile
    
    # Categorizzazione delle classi in base alla distribuzione
    rare_indices = np.where(val_class_distribution <= threshold_rare)[0]
    common_indices = np.where(val_class_distribution > threshold_common)[0]
    normal_indices = np.where((val_class_distribution > threshold_rare) & 
                            (val_class_distribution <= threshold_common))[0]
    
    print("\n=== VALUTAZIONE PER CATEGORIE DI FREQUENZA NEL VALIDATION SET ===")
    print(f"- Classi rare (≤{threshold_rare:.1f} esempi): {len(rare_indices)} classi")
    print(f"- Classi normali ({threshold_rare:.1f}-{threshold_common:.1f} esempi): {len(normal_indices)} classi")
    print(f"- Classi comuni (>{threshold_common:.1f} esempi): {len(common_indices)} classi")
    
    # Funzione per calcolare le metriche per un sottoinsieme di classi
    def calculate_metrics(class_indices, category_name):
        if not class_indices.size:
            print(f"Nessuna classe nella categoria {category_name}, metriche non calcolate.")
            return None, None, None, None
            
        # Estrae solo le colonne corrispondenti alle classi nella categoria
        y_true_subset = y_true[:, class_indices]
        y_pred_subset = y_pred[:, class_indices]
        y_pred_prob_subset = y_pred_prob[:, class_indices]
        
        # Precision, recall, F1 per ogni classe nella categoria
        precision = precision_score(y_true_subset, y_pred_subset, average=None, zero_division=0)
        recall = recall_score(y_true_subset, y_pred_subset, average=None, zero_division=0)
        f1 = f1_score(y_true_subset, y_pred_subset, average=None, zero_division=0)
        
        # Media delle metriche per questa categoria
        avg_precision = np.mean(precision)
        avg_recall = np.mean(recall)
        avg_f1 = np.mean(f1)
        
        # AUC-ROC per ogni classe (se ci sono sia positivi che negativi)
        aucs = []
        for i in range(y_true_subset.shape[1]):
            if len(np.unique(y_true_subset[:, i])) > 1:  # Verifica che ci siano sia 0 che 1
                auc = roc_auc_score(y_true_subset[:, i], y_pred_prob_subset[:, i])
                aucs.append(auc)
        
        avg_auc = np.mean(aucs) if aucs else float('nan')
        
        # Statistiche sul numero di classi e performance
        top_classes = []
        bottom_classes = []
        
        # Identifica le classi con le migliori e peggiori performance (per F1)
        if len(f1) > 0:
            sorted_idx = np.argsort(f1)
            
            # Prendi le 3 migliori e le 3 peggiori (o meno se non ci sono abbastanza classi)
            n_to_show = min(3, len(f1))
            
            for i in range(1, n_to_show + 1):
                if i <= len(sorted_idx):
                    # Classi con peggior performance
                    worst_idx = sorted_idx[i-1]
                    cls_name = all_species[class_indices[worst_idx]]
                    n_samples = val_class_distribution[class_indices[worst_idx]]
                    bottom_classes.append((cls_name, f1[worst_idx], precision[worst_idx], recall[worst_idx], n_samples))
                
                if i <= len(sorted_idx):
                    # Classi con miglior performance
                    best_idx = sorted_idx[-i]
                    cls_name = all_species[class_indices[best_idx]]
                    n_samples = val_class_distribution[class_indices[best_idx]]
                    top_classes.append((cls_name, f1[best_idx], precision[best_idx], recall[best_idx], n_samples))
        
        print(f"\n=== Metriche per classi {category_name} (n={len(class_indices)}) ===")
        print(f"Precision media: {avg_precision:.4f}")
        print(f"Recall medio: {avg_recall:.4f}")
        print(f"F1-Score medio: {avg_f1:.4f}")
        print(f"AUC-ROC medio: {avg_auc:.4f}")
        
        if top_classes:
            print("\nClassi con le migliori performance (F1):")
            for cls, f1_val, prec, rec, n_samples in top_classes:
                print(f"- {cls} ({n_samples:.0f} esempi nel validation): F1={f1_val:.4f}, Precision={prec:.4f}, Recall={rec:.4f}")
                
        if bottom_classes:
            print("\nClassi con le peggiori performance (F1):")
            for cls, f1_val, prec, rec, n_samples in bottom_classes:
                print(f"- {cls} ({n_samples:.0f} esempi nel validation): F1={f1_val:.4f}, Precision={prec:.4f}, Recall={rec:.4f}")
        
        return avg_precision, avg_recall, avg_f1, avg_auc
    
    # Calcola le metriche per ogni categoria
    metrics_rare = calculate_metrics(rare_indices, "RARE")
    metrics_normal = calculate_metrics(normal_indices, "NORMALI")
    metrics_common = calculate_metrics(common_indices, "COMUNI")
    
    # Crea visualizzazioni solo se abbiamo metriche valide per tutte le categorie
    valid_metrics = []
    categories = []
    
    if metrics_rare is not None and metrics_rare[0] is not None:
        valid_metrics.append(metrics_rare)
        categories.append('Rare')
    
    if metrics_normal is not None and metrics_normal[0] is not None:
        valid_metrics.append(metrics_normal)
        categories.append('Normali')
        
    if metrics_common is not None and metrics_common[0] is not None:
        valid_metrics.append(metrics_common)
        categories.append('Comuni')
    
    # Visualizza i risultati solo se abbiamo abbastanza dati
    if len(valid_metrics) >= 2:
        # Plot delle metriche per categoria
        fig, ax = plt.subplots(figsize=(12, 6))
        
        x = np.arange(len(categories))
        width = 0.2
        
        # Estrai le metriche per categorie valide
        precisions = [m[0] for m in valid_metrics]
        recalls = [m[1] for m in valid_metrics]
        f1_scores = [m[2] for m in valid_metrics]
        aucs = [m[3] for m in valid_metrics]
        
        ax.bar(x - 1.5*width, precisions, width, label='Precision')
        ax.bar(x - 0.5*width, recalls, width, label='Recall')
        ax.bar(x + 0.5*width, f1_scores, width, label='F1-Score')
        ax.bar(x + 1.5*width, aucs, width, label='AUC-ROC')
        
        ax.set_title('Performance per Categoria di Frequenza', fontsize=15)
        ax.set_xlabel('Categoria', fontsize=12)
        ax.set_ylabel('Score', fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels(categories)
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.7)
        
        # Aggiungi valori sopra le barre
        for i, v in enumerate(precisions):
            ax.text(i - 1.5*width, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
            
        for i, v in enumerate(recalls):
            ax.text(i - 0.5*width, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
            
        for i, v in enumerate(f1_scores):
            ax.text(i + 0.5*width, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
            
        for i, v in enumerate(aucs):
            ax.text(i + 1.5*width, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.show()
        
        # Grafico a radar per confronto diretto
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, polar=True)
        
        metrics_names = ['Precision', 'Recall', 'F1-Score', 'AUC-ROC']
        
        angles = np.linspace(0, 2*np.pi, len(metrics_names), endpoint=False).tolist()
        angles += angles[:1]  # Chiude il cerchio
        
        # Colori diversi per ogni categoria
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
        
        for i, category in enumerate(categories):
            values = list(valid_metrics[i])
            values += values[:1]  # Chiude il cerchio
            
            ax.plot(angles, values, 'o-', linewidth=2, label=category, color=colors[i % len(colors)])
            ax.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])
        
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        
        ax.set_thetagrids(np.degrees(angles[:-1]), metrics_names)
        ax.set_ylim(0, 1)
        ax.set_title('Confronto Performance per Categoria di Frequenza', fontsize=15, y=1.1)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
        
        plt.tight_layout()
        plt.show()
    else:
        print("Dati insufficienti per visualizzare il grafico di confronto.")


# Carica il miglior modello prima della valutazione
proceed_with_evaluation = True

# Verifica innanzitutto che model_save_path sia definito
try:
    best_model_path = model_save_path.replace('.pth', '_best.pth')
    print(f"\nCaricamento del miglior modello da {best_model_path} per valutazione...")
except NameError:
    print("ATTENZIONE: model_save_path non è definito. Impossibile determinare il percorso del miglior modello.")
    proceed_with_evaluation = False

# Procedi solo se abbiamo un percorso valido
if proceed_with_evaluation and os.path.exists(best_model_path):
    try:
        # Salva una copia del modello attuale
        current_model_state = model.state_dict().copy()
        
        # Carica il miglior modello
        checkpoint = torch.load(best_model_path, map_location=config.DEVICE)
        
        # Applica i pesi del miglior modello
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            best_epoch = checkpoint.get('epoch', 'N/A')
            best_val_loss = checkpoint.get('best_val_loss', 'N/A')
            print(f"Miglior modello caricato con successo (epoca {best_epoch}, val_loss: {best_val_loss})")
        else:
            model.load_state_dict(checkpoint)
            print("Miglior modello caricato con successo")
    except Exception as e:
        print(f"Errore durante il caricamento del modello: {e}")
        proceed_with_evaluation = False
else:
    print(f"ATTENZIONE: File del miglior modello non trovato.")
    proceed_with_evaluation = False

# Esegui la valutazione solo se abbiamo caricato correttamente il miglior modello
if proceed_with_evaluation:
    print("\nValutazione del modello sul validation set...")
    evaluate_model_on_validation(
        model=model,
        val_loader=val_loader,
        device=config.DEVICE
    )
else:
    print("\nValutazione saltata. Proseguire con la cella successiva.")

## 8. Generazione della Submission

## Generazione della Submission

Implementiamo la funzione per generare le predizioni sui file di test:
- Caricamento e segmentazione delle soundscape di test
- Estrazione di spettrogrammi Mel da ciascun segmento
- Generazione delle predizioni con il modello addestrato
- Creazione del file di submission nel formato richiesto dalla competizione

In [ ]:
def generate_submission_simple(model, device=config.DEVICE):
    """
    Genera un file di submission usando una segmentazione semplificata.
    
    Args:
        model: Modello PyTorch addestrato
        device: Device per inferenza ('cuda' o 'cpu')
        
    Returns:
        pd.DataFrame: DataFrame di submission
    """
    model.to(device)
    model.eval()
    
    # Set seed per riproducibilità
    np.random.seed(42)
    
    # Percorso dei test soundscapes
    test_soundscape_path = config.TEST_SOUNDSCAPES_DIR
    test_soundscapes = [os.path.join(test_soundscape_path, afile) 
                        for afile in sorted(os.listdir(test_soundscape_path)) 
                        if afile.endswith('.ogg')]
    
    print(f"Elaborazione di {len(test_soundscapes)} file soundscape...")
    
    # Crea DataFrame per le predizioni
    predictions = pd.DataFrame(columns=['row_id'] + all_species)
    
    for soundscape in tqdm(test_soundscapes, desc="Elaborazione soundscapes"):
        # Carica audio
        sig, rate = librosa.load(path=soundscape, sr=config.SR)
        
        # Split in segmenti da 5 secondi
        segment_length = rate * config.TEST_CLIP_DURATION
        chunks = []
        for i in range(0, len(sig), segment_length):
            chunk = sig[i:i+segment_length]
            # Padda se necessario
            if len(chunk) < segment_length:
                chunk = np.pad(chunk, (0, segment_length - len(chunk)), mode='constant')
            chunks.append(chunk)
        
        # Genera predizioni per ogni segmento
        for i, chunk in enumerate(chunks):
            # Calcola row_id (nome file + tempo finale del segmento in secondi)
            file_name = os.path.basename(soundscape).split('.')[0]
            row_id = f"{file_name}_{i * config.TEST_CLIP_DURATION + config.TEST_CLIP_DURATION}"
            
            # Calcola spettrogramma Mel
            mel_spec = librosa.feature.melspectrogram(
                y=chunk, sr=config.SR,
                n_fft=config.N_FFT,
                hop_length=config.HOP_LENGTH,
                n_mels=config.N_MELS,
                fmin=config.FMIN,
                fmax=config.FMAX
            )
            
            # Converti in scala logaritmica (dB) e normalizza
            log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
            min_val = np.min(log_mel_spec)
            max_val = np.max(log_mel_spec)
            if max_val > min_val:
                log_mel_spec = (log_mel_spec - min_val) / (max_val - min_val)
            else:
                log_mel_spec = np.zeros_like(log_mel_spec)
            
            # Aggiungi dimensione batch e canale
            log_mel_spec = np.expand_dims(np.expand_dims(log_mel_spec, axis=0), axis=0)
            
            # Converti in tensor
            input_tensor = torch.tensor(log_mel_spec, dtype=torch.float32).to(device)
            
            resize_transform = transforms.Resize((224, 224), 
                interpolation=transforms.InterpolationMode.BICUBIC)
            input_tensor = resize_transform(input_tensor)

            # Effettua predizione
            with torch.no_grad():
                output = model(input_tensor)
                scores = torch.sigmoid(output).cpu().numpy()[0]
            
            # Aggiungi riga al DataFrame di predizioni
            new_row = pd.DataFrame([[row_id] + list(scores)], columns=['row_id'] + all_species)
            predictions = pd.concat([predictions, new_row], axis=0, ignore_index=True)
    
    # Salva la submission come CSV
    predictions.to_csv("submission.csv", index=False)
    
    return predictions

# Genera submission
print("\nGenerazione del file di submission con il nuovo metodo...")
submission_df = generate_submission_simple(model)
    
if submission_df is not None:
    print("\nAnteprima del file di submission:")
    print(submission_df.head())
else:
    print("\nSalto la generazione della submission perché non siamo su Kaggle.")


Generazione del file di submission con il nuovo metodo...
Elaborazione di 0 file soundscape...


Elaborazione soundscapes: 0it [00:00, ?it/s]


Anteprima del file di submission:
Empty DataFrame
Columns: [row_id, 1139490, 1192948, 1194042, 126247, 1346504, 134933, 135045, 1462711, 1462737, 1564122, 21038, 21116, 21211, 22333, 22973, 22976, 24272, 24292, 24322, 41663, 41778, 41970, 42007, 42087, 42113, 46010, 47067, 476537, 476538, 48124, 50186, 517119, 523060, 528041, 52884, 548639, 555086, 555142, 566513, 64862, 65336, 65344, 65349, 65373, 65419, 65448, 65547, 65962, 66016, 66531, 66578, 66893, 67082, 67252, 714022, 715170, 787625, 81930, 868458, 963335, amakin1, amekes, ampkin1, anhing, babwar, bafibi1, banana, baymac, bbwduc, bicwre1, bkcdon, bkmtou1, blbgra1, blbwre1, blcant4, blchaw1, blcjay1, blctit1, blhpar1, blkvul, bobfly1, bobher1, brtpar1, bubcur1, bubwre1, bucmot3, bugtan, butsal1, cargra1, cattyr, chbant1, chfmac1, cinbec1, cocher1, cocwoo1, colara1, colcha1, compau, compot1, ...]
Index: []

[0 rows x 207 columns]
